In [29]:
pip install pyspark

Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: pyspark in c:\users\sridhar sudhakar\appdata\local\programs\python\python310\lib\site-packages (4.1.2)




[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:


from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import *

In [31]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import *

In [37]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("PySparkDemo") \
    .getOrCreate()

print(spark)

In [ ]:
df=spark.read.csv('/Volumes/pyspark/tables/data/orders.csv',header=True,inferSchema=True)
df.show()

In [0]:
df.describe().show()

+-------+--------+----------+-----------------+----------------+-------------+----------------+------------+
|summary|order_id|product_id|         quantity|      order_date|customer_city|    payment_mode|order_status|
+-------+--------+----------+-----------------+----------------+-------------+----------------+------------+
|  count|    1965|      2030|             1991|            2030|         2030|            2030|        2030|
|   mean|    NULL|      NULL|62.01703665462055|            NULL|         NULL|            NULL|        NULL|
| stddev|    NULL|      NULL|751.1703067691026|            NULL|         NULL|            NULL|        NULL|
|    min|     N/A|      P001|                -|01 December 2024|    Ahmedabad|CASH ON DELIVERY|   Cancelled|
|    max| ORD3000|      p101|             none|     Mar 01 2025|         pune|             upi|    Returned|
+-------+--------+----------+-----------------+----------------+-------------+----------------+------------+



In [0]:
df.select([count(when(col(c).isNull(),c)).alias(c) for c in df.columns]).show()

+--------+----------+--------+----------+-------------+------------+------------+
|order_id|product_id|quantity|order_date|customer_city|payment_mode|order_status|
+--------+----------+--------+----------+-------------+------------+------------+
|      65|         0|      39|         0|            0|           0|           0|
+--------+----------+--------+----------+-------------+------------+------------+



In [0]:
df= df.dropna()
df.select([count(when(col(c).isNull(),c)).alias(c) for c in df.columns]).show()

+--------+----------+--------+----------+-------------+------------+------------+
|order_id|product_id|quantity|order_date|customer_city|payment_mode|order_status|
+--------+----------+--------+----------+-------------+------------+------------+
|       0|         0|       0|         0|            0|           0|           0|
+--------+----------+--------+----------+-------------+------------+------------+



In [0]:
df= df.withColumn('order_id',lower('order_id'))\
    .withColumn('product_id',lower('product_id'))\
    .withColumn('customer_city',lower('customer_city'))\
    .withColumn('payment_mode',lower('payment_mode'))\
    .withColumn('order_status',lower('order_status'))
df.display()

order_id,product_id,quantity,order_date,customer_city,payment_mode,order_status
ord2263,p009,1,10-02-2025,delhi,cash on delivery,delivered
ord1298,p003,10,Dec 25 2024,pune,cashondelivery,delivered
ord1060,p002,8,02/05/2025,pune,net banking,cancelled
ord2697,p002,9,2024-12-13,ahmedabad,cash on delivery,delivered
ord1744,p008,2,Dec 07 2024,mumbai,net banking,delivered
ord2544,p033,3,05-01-2025,ahmedabad,cash on delivery,delivered
ord1977,p002,1,13/12/2024,delhi,upi,delivered
ord1353,p034,7,01/29/2025,mumbai,debitcard,delivered
ord2432,p033,2,22/01/2025,delhi,net banking,delivered
ord2402,p007,1,2024-12-18,bengaluru,net banking,cancelled


In [0]:
# formating product_id
df= df.withColumn(
    'product_id',
    lower(
        regexp_replace(
            trim(col('product_id')),'-',''
    )
)
)

df=df.withColumn('order_id',when(col('order_id')=='n/a',None).otherwise(col('order_id')))
df=df.dropna(subset="order_id")                 

# formatting quantity

df = (
    df.withColumn("quantity", regexp_replace(col("quantity"), "[^0-9-]", ""))
      .withColumn("quantity", expr("try_cast(quantity as int)"))
      .withColumn(
          "quantity",
          when(col("quantity") < 0, 0)
          .when(col("quantity") > 1000, 0)
          .otherwise(col("quantity"))
      )
      .fillna({"quantity": 0})
)


#formating order_date

df = df.withColumn(
    "order_date",
    coalesce(
        expr("try_to_date(order_date,'yyyy-MM-dd')"),
        expr("try_to_date(order_date,'dd-MM-yyyy')"),
        expr("try_to_date(order_date,'dd/MM/yyyy')"),
        expr("try_to_date(order_date,'MM-dd-yyyy')"),
        expr("try_to_date(order_date,'yyyy/MM/dd')"),
        expr("try_to_date(order_date,'MMM d yyyy')")
    )
)
df = df.withColumn("order_date", date_format(col("order_date"), "yyyy-MM-dd"))

df=df.fillna({'order_date':'2020-01-01'})


#  formating customer_city

df=df.withColumn(
    'customer_city',trim(col('customer_city')))\
    .withColumn(
      "customer_city",
    when(col("customer_city") == "madras", "chennai")
    .when(col("customer_city") == "calcutta", "kolkata")
    .when(col("customer_city") == "bombay", "mumbai")
    .when(col("customer_city") == "bengaluru", "bangalore")
    .otherwise(col("customer_city")))


 #formating payment mode

df=df.withColumn(
    'payment_mode',trim(col('payment_mode')))\
    .withColumn(
      "payment_mode",
    when(col("payment_mode") == "dc", "debit_card")
    .when(col("payment_mode") == "debit card", "debit_card")
    .when(col("payment_mode") == "debitcard", "debit_card")
    .when(col("payment_mode") == "u.p.i", "u_p_i")
    .when(col("payment_mode") == "upi", "u_p_i")
    .when(col("payment_mode") == "credit card", "credit_card")
    .when(col("payment_mode") == "creditcard", "credit_card")
    .when(col("payment_mode") == "cc", "credit_card")
    .when(col("payment_mode") == "cash on delivery", "u_p_i")
    .when(col("payment_mode") == "cashondelivery", "u_p_i")
    .when(col("payment_mode") == "cod", "u_p_i")
    .when(col("payment_mode") == "netbanking", "net_banking")
    .when(col("payment_mode") == "net banking", "net_banking")
    .otherwise(col("payment_mode")))

    

In [0]:
spark.sql("SHOW VOLUMES  IN pyspark.tables").show()


+--------+-----------+
|database|volume_name|
+--------+-----------+
|  tables|       data|
+--------+-----------+



In [0]:
df.write \
  .mode("overwrite") \
  .option("header", True) \
  .csv("/Volumes/pyspark/tables/data/cleaned_orders")

In [0]:
df2=spark.read.option('multiline','true').json('/Volumes/pyspark/tables/data/products.json')
df2.show()


+---------+----------+--------------------+----------+
| category|product_id|        product_name|unit_price|
+---------+----------+--------------------+----------+
|  Staples|      P001|       Tata Salt 1kg|      22.0|
|  Staples|      P002| Aashirvaad Atta 5kg|     275.0|
|  Staples|      P003|Fortune Sunflower...|     135.0|
|  Staples|      P004|India Gate Basmat...|     110.0|
|  Staples|      P005|       Toor Dal 500g|      68.0|
|    Dairy|      P006|Amul Full Cream M...|      68.0|
|    Dairy|      P007|    Amul Butter 500g|     285.0|
|    Dairy|      P008|Mother Dairy Curd...|      48.0|
|    Dairy|      P009|Amul Cheese Slice...|     135.0|
|    Dairy|      P010| Nandini Paneer 200g|      90.0|
|   Snacks|      P011|Lay's Classic Sal...|      20.0|
|   Snacks|      P012|Bingo Mad Angles 90g|      35.0|
|   Snacks|      P013|Britannia Good Da...|      30.0|
|   Snacks|      P014|Haldiram Bhujia 200g|      80.0|
|   Snacks|      P015|Parle-G Biscuits ...|      65.0|
|Beverages

In [0]:
df2.printSchema()

root
 |-- category: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- unit_price: double (nullable = true)



In [0]:
df2.select([count(when(col(c).isNull(),c)).alias(c) for c in df2.columns]).show()

+--------+----------+------------+----------+
|category|product_id|product_name|unit_price|
+--------+----------+------------+----------+
|       0|         0|           0|         0|
+--------+----------+------------+----------+



In [0]:
df2.select('product_id').distinct().display()

product_id
p010
p036
p011
p020
p028
p016
p001
p024
p039
p025


In [0]:
df2= df2.withColumn('category',lower('category'))\
    .withColumn('product_id',lower('product_id'))\
    .withColumn('product_name',lower('product_name'))
df2.display()

category,product_id,product_name,unit_price
staples,p001,tata salt 1kg,22.0
staples,p002,aashirvaad atta 5kg,275.0
staples,p003,fortune sunflower oil 1l,135.0
staples,p004,india gate basmati 1kg,110.0
staples,p005,toor dal 500g,68.0
dairy,p006,amul full cream milk 1l,68.0
dairy,p007,amul butter 500g,285.0
dairy,p008,mother dairy curd 400g,48.0
dairy,p009,amul cheese slices 200g,135.0
dairy,p010,nandini paneer 200g,90.0


In [0]:
df2.write \
  .mode("overwrite") \
  .option("header", True) \
  .csv("/Volumes/pyspark/tables/data/cleaned_products")

In [0]:
df_merged = df.join(df2, on="product_id", how="inner")
df_merged.show()

+----------+--------+--------+----------+-------------+----------------+------------+-------------+--------------------+----------+
|product_id|order_id|quantity|order_date|customer_city|    payment_mode|order_status|     category|        product_name|unit_price|
+----------+--------+--------+----------+-------------+----------------+------------+-------------+--------------------+----------+
|      p009| ord2263|       1|2025-02-10|        delhi|           u_p_i|   delivered|        dairy|amul cheese slice...|     135.0|
|      p003| ord1298|      10|2024-12-25|         pune|           u_p_i|   delivered|      staples|fortune sunflower...|     135.0|
|      p002| ord1060|       8|2025-05-02|         pune|     net_banking|   cancelled|      staples| aashirvaad atta 5kg|     275.0|
|      p002| ord2697|       9|2024-12-13|    ahmedabad|           u_p_i|   delivered|      staples| aashirvaad atta 5kg|     275.0|
|      p008| ord1744|       2|2024-12-07|       mumbai|     net_banking|   d

In [0]:
df_merged.select([count(when(col(c).isNull(),c)).alias(c) for c in df_merged.columns]).show()

+----------+--------+--------+----------+-------------+------------+------------+--------+------------+----------+
|product_id|order_id|quantity|order_date|customer_city|payment_mode|order_status|category|product_name|unit_price|
+----------+--------+--------+----------+-------------+------------+------------+--------+------------+----------+
|         0|       0|       0|         0|            0|           0|           0|       0|           0|         0|
+----------+--------+--------+----------+-------------+------------+------------+--------+------------+----------+



In [0]:
df_merged.write \
  .mode("overwrite") \
  .option("header", True) \
  .csv("/Volumes/pyspark/tables/data/cleaned_orders_products")

### QUERIES

Q1	What is the total revenue generated across all orders this period?
Hint: Use .sum() on the total_revenue column.
__

In [0]:
df_merged = df_merged.withColumn("total_revenue", col("quantity") * col("unit_price"))
df_merged.filter(col("order_status") == "delivered") \
  .withColumn("total_revenue", col("quantity") * col("unit_price")) \
  .select(sum("total_revenue")) \
  .show()

+------------------+
|sum(total_revenue)|
+------------------+
|          479504.0|
+------------------+



Q2	Which product category contributed the highest total revenue?
Hint: Use .groupby('category')['total_revenue'].sum() and find the top entry.


In [0]:
df_grouped = df_merged.groupBy("category") \
               .agg(sum("total_revenue").alias("total_revenue"))
df_grouped.orderBy(col("total_revenue").desc()).show(1)               

+--------+-------------+
|category|total_revenue|
+--------+-------------+
| staples|     210870.0|
+--------+-------------+
only showing top 1 row


Q3	Which product was ordered the most by total quantity sold?
Hint: Use .groupby('product_name')['quantity'].sum().idxmax()


In [0]:
df_merged.groupBy("product_name") \
  .agg(sum("quantity").alias("total_quantity")) \
  .orderBy(col("total_quantity").desc()) \
  .show(1)

+-----------------+--------------+
|     product_name|total_quantity|
+-----------------+--------------+
|maggi noodles 70g|           531|
+-----------------+--------------+
only showing top 1 row


Q4	Which city placed the highest number of orders?
Hint: Use .groupby('city')['order_id'].count().idxmax()


In [0]:
df_merged.groupBy("customer_city") \
  .agg(count("order_id").alias("total_orders")) \
  .orderBy(col("total_orders").desc()) \
  .show(1)

+-------------+------------+
|customer_city|total_orders|
+-------------+------------+
|      kolkata|         255|
+-------------+------------+
only showing top 1 row


Q5	How many orders had a quantity of zero after null-filling, and what percentage of total orders is that?
Hint: Filter where quantity == 0, then calculate (count / total_rows) * 100


In [0]:
total_orders = df_merged.count()
zero_qty_orders = df_merged.filter(col("quantity") == 0).count()

percentage = (zero_qty_orders / total_orders) * 100

print("Zero Quantity Orders:", zero_qty_orders)
print("Total Orders:", total_orders)
print("Percentage:", percentage)

Zero Quantity Orders: 104
Total Orders: 1876
Percentage: 5.543710021321962


Q6	What is the average order value (average total_revenue per order)?
Hint: Use total_revenue.mean() on the merged DataFrame.


In [0]:
df_merged.withColumn("total_revenue", col("quantity") * col("unit_price")) \
  .select(avg("total_revenue").alias("avg_order_value")) \
  .show()

+-----------------+
|  avg_order_value|
+-----------------+
|452.9749466950959|
+-----------------+

